# Neuronas alternativas

Neuronas flotantes (palabras como neuronas en el espacio), redes semánticas, árbol fonológico, neuronas con un clasificador clásico dentro (KNN, árbol de decisión, Naive Bayes) y redes genéticas.

Parte del **laboratorio neuronal**. Cada sección es un experimento independiente: ejecuta solo la que te interese, de arriba hacia abajo (cada una instala lo que necesita).

> **Medido en el banco** (`laboratorio/banco/`, 5 semillas, datos de prueba): ver `laboratorio/banco/README.md`, sección «cuaderno 03». La neurona KNN, los clasificadores clásicos y las redes genéticas están ahí. Las redes genéticas tal cual se derrumban (reemplazan neuronas vivas por copias de neuronas muertas); con el criterio invertido funcionan igual que la red estándar.

## Neuronas flotantes

Cada palabra es una neurona con su vector BERT y una posición en el plano (t-SNE). La celda siguiente genera el archivo `palabras_con_embedding_y_posicion.pkl` que usan las demás.

In [ ]:
pip install transformers

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch
import numpy as np
from sklearn.manifold import TSNE
import pandas as pd
import os # Import the os module

# --- Configuración del modelo BERT ---
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
model.eval()

# --- Frase de prueba ---
sentence = "The house is big and has a beautiful garden with flowers."

# --- Tokenización ---
inputs = tokenizer(sentence, return_tensors="pt")
with torch.no_grad():
    outputs = model(**inputs)

# Extraemos los embeddings de la última capa oculta (batch_size, seq_len, hidden_size)
embeddings = outputs.last_hidden_state.squeeze(0)  # (seq_len, hidden_size)
tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'].squeeze(0))

# --- Filtrado: removemos [CLS], [SEP] y tokens subword (##)
valid_data = [
    (token, embeddings[i].numpy())
    for i, token in enumerate(tokens)
    if token not in ["[CLS]", "[SEP]"] and not token.startswith("##")
]

words, vectors = zip(*valid_data)

# --- Reducción a 2D con t-SNE ---
# Adjust perplexity if the number of samples is small
perplexity_value = min(len(vectors) - 1, 30) # Default perplexity is 30, but needs to be < n_samples
if perplexity_value < 1: # Perplexity must be greater than 1
    print("Warning: Not enough data points to perform t-SNE with perplexity > 1. Skipping t-SNE.")
    # Handle the case where t-SNE is skipped, perhaps by assigning default positions or raising an error
    positions_2d = np.zeros((len(vectors), 2)) # Assign dummy positions if skipping
else:
    tsne = TSNE(n_components=2, random_state=0, perplexity=perplexity_value)
    positions_2d = tsne.fit_transform(np.stack(vectors))


# --- Estructura final: palabra, x, y, vector768 ---
records = []
# Ensure that all lists for zip have the same length, especially if t-SNE was skipped
if len(words) == len(positions_2d) == len(vectors):
    for word, pos, vec in zip(words, positions_2d, vectors):
        record = {
            "word": word,
            "x": pos[0],
            "y": pos[1],
            "vector768": vec
        }
        records.append(record)
else:
    print("Error: Mismatch in list lengths after data processing. Cannot create records.")


# --- Guardar a archivo ---
df = pd.DataFrame(records)

# Define the directory path
output_dir = "/content/DJ"
# Create the directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# Define the full file paths
pickle_path = os.path.join(output_dir, "palabras_con_embedding_y_posicion.pkl")
csv_path = os.path.join(output_dir, "palabras_con_posicion.csv")

# Save to pickle
df.to_pickle(pickle_path)  # formato binario eficiente
print(f"Saved pickle file to: {pickle_path}")

# Also save as CSV if desired (without full vector)
df_simple = df.drop(columns=["vector768"])
df_simple.to_csv(csv_path, index=False)
print(f"Saved CSV file to: {csv_path}")

pickle_path, csv_path

In [ ]:
import pandas as pd
import torch
import numpy as np
import matplotlib.pyplot as plt
import torch.nn.functional as F

# --- Hiperparámetros (algunos se vuelven redundantes o cambian su significado) ---
# Ya no necesitamos definir NUM_PROCESSING_NEURONS directamente, se calcula
# Ya no necesitamos rangos aleatorios para las posiciones de procesamiento
ACTIVATION_THRESHOLD = 0.5
OUTPUT_SCALE_FACTOR = 5
USE_COSINE_SIMILARITY = True
USE_SIGMOID_ACTIVATION = False

# --- Clase de neurona flotante (sin cambios) ---
class FloatingNeuron:
    def __init__(self, ID, position, vector768, neuron_type='input'):
        self.id = ID
        self.x, self.y = position
        self.vector768 = vector768
        self.type = neuron_type

    def __repr__(self):
        return f"Neuron[{self.id}] (Type: {self.type}) @ ({self.x:.2f}, {self.y:.2f})"

# --- Cargar datos de neuronas de entrada desde archivos (sin cambios) ---
def load_input_neurons_from_files(pkl_path):
    data = pd.read_pickle(pkl_path)
    neurons = []
    for row in data.itertuples():
        neuron = FloatingNeuron(
            ID=row.word,
            position=(row.x, row.y),
            vector768=row.vector768,
            neuron_type='input'
        )
        neurons.append(neuron)
    return neurons

# --- MODIFICADO: Función para generar neuronas de procesamiento con la nueva lógica ---
def generate_processing_neurons_v2(input_neurons, avg_vector_dim=768):
    processing_neurons = []

    for input_neuron in input_neurons:
        # Vector base para la neurona de procesamiento.
        # Aquí, aleatorio. En un modelo entrenado, sería algo aprendido.
        # Para el ejemplo, podríamos incluso usar una variación del vector de la neurona de entrada
        # para que tengan alguna relación semántica desde el inicio.
        base_proc_vector = np.random.rand(avg_vector_dim) * 2 - 1

        # --- Primera neurona de procesamiento conectada a esta neurona de entrada ---
        # ID: basado en la neurona de entrada y su multiplicador de posición
        proc_id_1 = f"Proc_{input_neuron.id}_pos2"
        # Posición: X*2, Y*2
        proc_pos_1 = (input_neuron.x * 2, input_neuron.y * 2)

        neuron_1 = FloatingNeuron(
            ID=proc_id_1,
            position=proc_pos_1,
            vector768=base_proc_vector, # Este vector será el "peso" de esta neurona de procesamiento
            neuron_type='processing'
        )
        processing_neurons.append(neuron_1)

        # --- Segunda neurona de procesamiento conectada a esta neurona de entrada ---
        proc_id_2 = f"Proc_{input_neuron.id}_neg2"
        # Posición: X*-2, Y*-2
        proc_pos_2 = (input_neuron.x * -2, input_neuron.y * -2)

        neuron_2 = FloatingNeuron(
            ID=proc_id_2,
            position=proc_pos_2,
            vector768=np.random.rand(avg_vector_dim) * 2 - 1, # Otro vector aleatorio
            neuron_type='processing'
        )
        processing_neurons.append(neuron_2)

    return processing_neurons

# --- map_output_to_word (sin cambios) ---
def map_output_to_word(activation_vector, input_neurons):
    max_sim = -float('inf')
    best_word = None

    if isinstance(activation_vector, np.ndarray):
        activation_vector = torch.tensor(activation_vector, dtype=torch.float32)
    elif not isinstance(activation_vector, torch.Tensor):
        raise TypeError("activation_vector debe ser numpy array o torch tensor")

    if activation_vector.dim() == 0:
        print("Advertencia: activation_vector es un escalar. map_output_to_word necesita un vector de embeddings.")
        return None, None # Devuelve None para ambos

    for neuron in input_neurons:
        if neuron.vector768 is None:
            continue

        neuron_vec_tensor = torch.tensor(neuron.vector768, dtype=torch.float32)

        sim = F.cosine_similarity(
            activation_vector.unsqueeze(0),
            neuron_vec_tensor.unsqueeze(0)
        )
        sim_val = sim.item()

        if sim_val > max_sim:
            max_sim = sim_val
            best_word = neuron.id

    return best_word, max_sim

# --- visualize_all_neurons (sin cambios, ya maneja diferentes tipos) ---
def visualize_all_neurons(all_neurons):
    if not all_neurons:
        print("No hay neuronas para visualizar.")
        return

    x_coords = [n.x for n in all_neurons]
    y_coords = [n.y for n in all_neurons]
    ids = [n.id for n in all_neurons]
    types = [n.type for n in all_neurons]

    color_map = {'input': 'blue', 'processing': 'red'}
    colors = [color_map[t] for t in types]

    plt.figure(figsize=(14, 12))
    plt.scatter(x_coords, y_coords, c=colors, s=150, alpha=0.7, edgecolors='w', linewidth=0.5)

    for i, ID in enumerate(ids):
        plt.annotate(
            ID,
            (x_coords[i], y_coords[i]),
            textcoords="offset points",
            xytext=(5,5),
            ha='left', va='bottom',
            fontsize=9, color='dimgray' if types[i] == 'input' else 'black'
        )

    plt.title('Posición 2D de Neuronas Flotantes (Input y Procesamiento)', fontsize=16)
    plt.xlabel('Coordenada X', fontsize=12)
    plt.ylabel('Coordenada Y', fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.axhline(0, color='grey', linewidth=0.5)
    plt.axvline(0, color='grey', linewidth=0.5)

    from matplotlib.lines import Line2D
    legend_elements = [
        Line2D([0], [0], marker='o', color='w', label='Input (Palabra)', markerfacecolor='blue', markersize=10),
        Line2D([0], [0], marker='o', color='w', label='Procesamiento', markerfacecolor='red', markersize=10)
    ]
    plt.legend(handles=legend_elements, loc='best', title="Tipo de Neurona")
    plt.show()

# --- Simulación de Activación y Salida (sin cambios, ya usa hiperparámetros) ---
def simulate_neuron_activation(input_word_embedding, processing_neurons):
    results = []
    input_word_embedding_tensor = torch.tensor(input_word_embedding, dtype=torch.float32).unsqueeze(0)

    for proc_neuron in processing_neurons:
        proc_neuron_vector_tensor = torch.tensor(proc_neuron.vector768, dtype=torch.float32).unsqueeze(0)

        if USE_COSINE_SIMILARITY:
            raw_activation = F.cosine_similarity(input_word_embedding_tensor, proc_neuron_vector_tensor).item()
        else:
            raw_activation = torch.dot(input_word_embedding_tensor.squeeze(0), proc_neuron_vector_tensor.squeeze(0)).item()

        if USE_SIGMOID_ACTIVATION:
            activation_output = torch.sigmoid(torch.tensor(raw_activation)).item()
        else:
            if raw_activation < ACTIVATION_THRESHOLD:
                activation_output = 0.0
            else:
                activation_output = raw_activation * OUTPUT_SCALE_FACTOR

        results.append({
            'processing_neuron_id': proc_neuron.id,
            'raw_activation_score': raw_activation,
            'output_activation': activation_output
        })
    return results

# --- Parte principal del script ---
if __name__ == "__main__":
    pkl_path = "/content/DJ/palabras_con_embedding_y_posicion.pkl"

    # 1. Cargar neuronas de entrada (palabras)
    input_neurons = load_input_neurons_from_files(pkl_path)
    print("Neuronas de entrada cargadas (primeras 5):")
    for n in input_neurons[:5]:
        print(n)

    # 2. Generar neuronas de procesamiento CON LA NUEVA LÓGICA
    processing_neurons = generate_processing_neurons_v2(input_neurons)

    print(f"\nNeuronas de procesamiento generadas ({len(processing_neurons)} en total, el doble de las de entrada):")
    for n in processing_neurons:
        print(n)

    # 3. Combinar todas las neuronas para visualización
    all_neurons = input_neurons + processing_neurons
    print("\nVisualizando todas las neuronas...")
    visualize_all_neurons(all_neurons)

    # 4. Simular un Input y Activación (con mejoras)
    print("\n--- Simulación de Activación de Neuronas de Procesamiento ---")

    if input_neurons:
        # Tomar una palabra de ejemplo para simular la entrada
        word_to_simulate = next((n for n in input_neurons if n.id.lower() == 'casa'), None)
        if word_to_simulate is None and len(input_neurons) > 0:
            word_to_simulate = input_neurons[0] # Usa la primera si 'casa' no existe

        if word_to_simulate:
            print(f"\nInput a la red de procesamiento: '{word_to_simulate.id}' (Embedding de la palabra de entrada)")

            activation_results = simulate_neuron_activation(
                word_to_simulate.vector768,
                processing_neurons
            )

            print("\nResultados de Activación de Neuronas de Procesamiento:")
            active_results = [r for r in activation_results if r['output_activation'] > 0]
            if active_results:
                for result in active_results:
                    print(f"- {result['processing_neuron_id']}: Similitud/Activación Cruda={result['raw_activation_score']:.4f}, Salida={result['output_activation']:.4f}")
            else:
                print("Ninguna neurona de procesamiento se activó por encima del umbral.")

            combined_output_vector = np.zeros(768)
            total_output_weight = 0

            for i, result in enumerate(activation_results):
                if result['output_activation'] > 0:
                    combined_output_vector += processing_neurons[i].vector768 * result['output_activation']
                    total_output_weight += result['output_activation']

            if total_output_weight > 0:
                combined_output_vector /= total_output_weight
                print(f"\nVector de salida combinado de las neuronas de procesamiento.")

                predicted_word, similarity = map_output_to_word(combined_output_vector, input_neurons)
                print(f"Predicción de la siguiente palabra (basada en neuronas de procesamiento): '{predicted_word}' (Similitud: {similarity:.4f})")
            else:
                print("\nNo se pudo generar un vector de salida combinado (ninguna neurona de procesamiento se activó).")
        else:
            print("No se encontró la palabra para simular o no hay neuronas de entrada.")

    # 5. Simulación de 'The house' (Mejorada para composición)
    print("\n--- Simulación de Composición: 'The house' ---")
    the_neuron = next((n for n in input_neurons if n.id.lower() == 'the'), None)
    house_neuron = next((n for n in input_neurons if n.id.lower() == 'house'), None)

    if the_neuron and house_neuron:
        combo_embedding = (the_neuron.vector768 + house_neuron.vector768) / 2.0

        print(f"Embedding combinado para 'The house' (promedio).")

        activation_results_combo = simulate_neuron_activation(combo_embedding, processing_neurons)

        print("\nResultados de Activación para el Embedding Combinado:")
        active_results_combo = [r for r in activation_results_combo if r['output_activation'] > 0]
        if active_results_combo:
            for result in active_results_combo:
                print(f"- {result['processing_neuron_id']}: Similitud/Activación Cruda={result['raw_activation_score']:.4f}, Salida={result['output_activation']:.4f}")
        else:
            print("Ninguna neurona de procesamiento se activó para el embedding combinado.")

        combined_output_vector_combo = np.zeros(768)
        total_output_weight_combo = 0

        for i, result in enumerate(activation_results_combo):
            if result['output_activation'] > 0:
                combined_output_vector_combo += processing_neurons[i].vector768 * result['output_activation']
                total_output_weight_combo += result['output_activation']

        if total_output_weight_combo > 0:
            combined_output_vector_combo /= total_output_weight_combo
            predicted_word_combo, similarity_combo = map_output_to_word(combined_output_vector_combo, input_neurons)
            print(f"\nPredicción de la siguiente palabra/concepto para 'The house': '{predicted_word_combo}' (Similitud: {similarity_combo:.4f})")
        else:
            print("\nNo se pudo generar un vector de salida combinado para 'The house'.")

    else:
        print("Las palabras 'The' o 'house' no se encontraron en las neuronas de entrada cargadas.")

In [ ]:
def generate_output_neurons(sequence_words, base_neurons, avg_vector_dim=768):
    output_neurons = []

    for i, word in enumerate(sequence_words):
        # Buscar neurona base con el mismo ID, o usar la primera si no existe
        base = next((n for n in base_neurons if n.id == word), base_neurons[0])

        # Crear una neurona de salida basada en esa posición
        out_id = f"Out_{word}"
        out_pos = (base.x * 3, base.y * 3)  # Más alejada para que no se superponga

        out_vector = np.random.rand(avg_vector_dim) * 2 - 1  # Vector aleatorio

        neuron = FloatingNeuron(
            ID=out_id,
            position=out_pos,
            vector768=out_vector,
            neuron_type='output'
        )
        output_neurons.append(neuron)

    return output_neurons

color_map = {'input': 'blue', 'processing': 'red', 'output': 'orange'}


## Red semántica (palabras → conceptos)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import networkx as nx

# Diccionario semántico: Neuronas de entrada → Neuronas abstractas
semantic_map = {
    "rey": "realeza",
    "reina": "realeza",
    "emperador": "realeza",
    "soldado": "militar",
    "general": "militar",
    "profesor": "educación",
    "estudiante": "educación"
}

# Lista única de entradas y categorías
input_neurons = list(semantic_map.keys())
processing_neurons = list(set(semantic_map.values()))

# Índices
input_idx = {name: i for i, name in enumerate(input_neurons)}
processing_idx = {name: i for i, name in enumerate(processing_neurons)}

# Dimensiones
input_size = len(input_neurons)
processing_size = len(processing_neurons)

# Matriz de conexiones semánticas
connection_matrix = torch.zeros(processing_size, input_size)
for inp, proc in semantic_map.items():
    i = processing_idx[proc]
    j = input_idx[inp]
    connection_matrix[i, j] = 1.0  # Conexión semántica (peso = 1)

class SemanticNetwork(nn.Module):
    def __init__(self, connection_matrix):
        super().__init__()
        self.connection_matrix = nn.Parameter(connection_matrix.clone(), requires_grad=False)

    def forward(self, x):
        # x: [batch, input_size]
        # Multiplicación lineal semántica: activa neuronas abstractas si sus entradas conectadas se activan
        y = torch.matmul(x, self.connection_matrix.T)
        return torch.sigmoid(y)  # activación de neuronas abstractas

# Red
net = SemanticNetwork(connection_matrix)

# Entrada: activamos solo "rey" y "emperador"
input_vector = torch.zeros((1, input_size))
input_vector[0, input_idx["rey"]] = 1.0
input_vector[0, input_idx["emperador"]] = 1.0

# Forward
output = net(input_vector)

# Mostrar activaciones
print("Activaciones de entrada:")
for name, idx in input_idx.items():
    print(f"{name}: {input_vector[0, idx].item():.2f}")

print("\nActivaciones de neuronas de procesamiento (semánticas):")
for name, idx in processing_idx.items():
    print(f"{name}: {output[0, idx].item():.2f}")

# Visualización con NetworkX
def draw_network(input_vector, output_vector):
    G = nx.DiGraph()
    pos = {}
    labels = {}

    # Neuronas de entrada
    for i, name in enumerate(input_neurons):
        G.add_node(name, layer="input")
        pos[name] = (0, -i)
        labels[name] = f"{name} ({input_vector[0, i].item():.0f})"

    # Neuronas de procesamiento
    for i, name in enumerate(processing_neurons):
        G.add_node(name, layer="processing")
        pos[name] = (2, -i)
        labels[name] = f"{name} ({output_vector[0, i].item():.2f})"

    # Conexiones semánticas
    for inp, proc in semantic_map.items():
        G.add_edge(inp, proc)

    # Dibujo
    plt.figure(figsize=(10, 6))
    nx.draw(G, pos, with_labels=True, labels=labels,
            node_color='skyblue', edge_color='gray',
            node_size=3000, font_size=10, font_weight='bold')
    plt.title("Red Semántica de Neuronas")
    plt.show()

draw_network(input_vector, output)

In [ ]:
# Árbol fonético-semántico neuronal básico (PyTorch)
import torch
import torch.nn as nn
import torch.nn.functional as F

# 1. Palabras y sus clases semánticas
palabras = {
    "abuelo": "familia",
    "abismo": "profundidad",
    "ebanista": "oficio",
    "ebriedad": "estado",
    "rey": "realeza",
    "reina": "realeza",
    "gato": "animal",
    "perro": "animal",
}

# 2. Tokenización por letras
def tokenizar(palabra):
    return list(palabra)

# 3. Mapeo de letras a IDs
letras = sorted(set("".join(palabras.keys())))
letra2idx = {l: i for i, l in enumerate(letras)}
idx2letra = {i: l for l, i in letra2idx.items()}

# 4. Mapeo de clases
clases = sorted(set(palabras.values()))
clase2idx = {c: i for i, c in enumerate(clases)}
idx2clase = {i: c for c, i in clase2idx.items()}

# 5. Modelos
embedding_dim = 16
hidden_dim = 32
num_clases = len(clases)

embedding = nn.Embedding(len(letra2idx), embedding_dim)

class NodoJerarquico(nn.Module):
    def __init__(self, embedding_dim, hidden_dim):
        super().__init__()
        self.cell = nn.GRUCell(embedding_dim, hidden_dim)
        self.hidden_dim = hidden_dim

    def forward(self, letras):
        h = torch.zeros(1, self.hidden_dim)
        for letra in letras:
            emb = embedding(torch.tensor([letra2idx[letra]]))
            h = self.cell(emb, h)
        return h

class ClasificadorSemantico(nn.Module):
    def __init__(self, hidden_dim, num_clases):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(hidden_dim, 32),
            nn.ReLU(),
            nn.Linear(32, num_clases)
        )

    def forward(self, h):
        return self.mlp(h)

# Instancia de modelos
nodo = NodoJerarquico(embedding_dim, hidden_dim)
clasificador = ClasificadorSemantico(hidden_dim, num_clases)

# Optimización
params = list(embedding.parameters()) + list(nodo.parameters()) + list(clasificador.parameters())
optimizer = torch.optim.Adam(params, lr=0.01)
loss_fn = nn.CrossEntropyLoss()

# 6. Entrenamiento
for epoch in range(100):
    total_loss = 0
    for palabra, clase in palabras.items():
        letras = tokenizar(palabra)
        h = nodo(letras)
        salida = clasificador(h)
        y = torch.tensor([clase2idx[clase]])
        loss = loss_fn(salida, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    if epoch % 10 == 0:
        print(f"Epoch {epoch:03d} | Loss: {total_loss:.4f}")

# 7. Pruebas
def clasificar_palabra(palabra):
    letras = tokenizar(palabra)
    h = nodo(letras)
    salida = clasificador(h)
    pred = torch.argmax(salida).item()
    return idx2clase[pred]

print("\n--- Pruebas ---")
for palabra in palabras.keys():
    pred = clasificar_palabra(palabra)
    print(f"{palabra:<10} → {pred}")

## Árbol fonológico (letras → sílabas → palabras)

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import networkx as nx

# ==== Nivel 0: Letras ====
letras = list("abcdefghijklmnñopqrstuvwxyz")
embedding_dim = 8

letra2idx = {l: i for i, l in enumerate(letras)}
idx2letra = {i: l for l, i in letra2idx.items()}
embedding = nn.Embedding(len(letras), embedding_dim)

# ==== Grafo ====
G = nx.DiGraph()

# Agregar nodos de letras (nivel 0)
for letra in letras:
    G.add_node(letra, tipo="letra")

# ==== Nivel 1: Sílabas ====
vocales = set('aeiouáéíóúü') # Incluir vocales con tilde y diéresis para mayor realismo
consonantes = [l for l in letras if l not in vocales]

silabas_validas = set() # Usaremos un set para almacenar sílabas únicas que construiremos
# Puedes añadir más reglas de silabificación si quieres. Aquí un ejemplo simple:
# Sílabas tipo CV (consonante + vocal)
for c in consonantes:
    for v in vocales:
        silaba = c + v
        silabas_validas.add(silaba)
# Sílabas tipo VC (vocal + consonante)
for v in vocales:
    for c in consonantes:
        silaba = v + c
        silabas_validas.add(silaba)
# Sílabas de una sola vocal
for v in vocales:
    silabas_validas.add(v)

# Agregar nodos de sílabas (nivel 1)
for silaba in silabas_validas:
    G.add_node(silaba, tipo="silaba")
    # Conectar sílaba con sus letras base
    for char in silaba:
        if char in letras: # Asegurarse de que el carácter es una letra del abecedario
            G.add_edge(char, silaba)

# ==== Nivel 2: Palabras ====
# Para un modelo más robusto, se usaría una biblioteca de silabificación real para español.
# Aquí, para el ejemplo, descomponemos manualmente.
palabras_descompuestas = {
    "luna": ["lu", "na"],
    "sol": ["sol"], # 'sol' se considera una sílaba única
    "casa": ["ca", "sa"],
    "mesa": ["me", "sa"],
    "rio": ["ri", "o"], # "o" como sílaba vocal
    "perro": ["pe", "rro"], # Ejemplo de doble consonante
    "arbol": ["ar", "bol"],
    "cielo": ["cie", "lo"]
}

# Agrega nodos de palabras y sus conexiones
for palabra, silabas_palabra in palabras_descompuestas.items():
    G.add_node(palabra, tipo="palabra")
    for silaba_componente in silabas_palabra:
        # Asegúrate de que la sílaba componente exista en tus silabas_validas
        # Si no existe (ej. "sol" o "rro" que no son CV/VC simples), puedes decidir cómo manejarlas:
        # 1. Añadirlas a silabas_validas si son sílabas genuinas pero no "CV/VC"
        # 2. Conectar las letras directamente a la palabra si la sílaba no es "estándar" (menos recomendable para esta jerarquía)
        if silaba_componente not in silabas_validas:
             silabas_validas.add(silaba_componente) # Añadir sílaba "no estándar" para que tenga nodo
             G.add_node(silaba_componente, tipo="silaba")
             for char in silaba_componente:
                 if char in letras:
                     G.add_edge(char, silaba_componente)

        G.add_edge(silaba_componente, palabra) # Conectar sílaba a palabra

# ==== Visualización ====

def visualizar_arbol_jerarquico(grafo):
    pos = {}
    # Podemos intentar organizar en capas, o con diseño spring para ver las conexiones
    # Para visualizar capas jerárquicas:
    levels = {
        "letra": 0,
        "silaba": 1,
        "palabra": 2
    }

    # Crear un diseño spring_layout para ver las relaciones, o un "layered" si hay un paquete
    # Para este ejemplo, un diseño spring puede mostrar mejor la interconexión.
    # Alternativamente, si quieres algo más estructurado como capas:

    # Intento de layout en capas (esto puede ser complicado para redes muy grandes)
    # nodes_by_level = {level: [] for level in levels.values()}
    # for n, data in grafo.nodes(data=True):
    #     nodes_by_level[levels[data['tipo']]].append(n)

    # # Posiciones para cada nivel (ejemplo simple, necesitaría más ajuste para estética)
    # for level, nodes in nodes_by_level.items():
    #     if nodes:
    #         y_pos = level * 10 # Separación vertical entre niveles
    #         x_step = 20 / len(nodes)
    #         for i, node in enumerate(nodes):
    #             pos[node] = (i * x_step - (len(nodes) * x_step / 2), y_pos)

    # Optamos por un diseño spring_layout para mejor visualización de todas las conexiones
    pos = nx.spring_layout(grafo, k=0.8, iterations=50) # k ajusta la distancia, iterations la calidad

    plt.figure(figsize=(16, 16)) # Aumentar tamaño para mejor visibilidad

    node_colors = []
    for n in G.nodes:
        if G.nodes[n]['tipo'] == 'letra':
            node_colors.append('lightblue')
        elif G.nodes[n]['tipo'] == 'silaba':
            node_colors.append('lightgreen')
        elif G.nodes[n]['tipo'] == 'palabra':
            node_colors.append('lightcoral')

    nx.draw(G, pos, with_labels=True, node_color=node_colors,
            node_size=1800, font_size=9, font_weight='bold', edge_color='gray', arrows=True, arrowsize=15)
    plt.title("Árbol neuronal jerárquico: Letras → Sílabas → Palabras", size=18)
    plt.show()

visualizar_arbol_jerarquico(G)

## Neuronas en un espacio 3D (conexiones por distancia)

In [ ]:
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# --- 1. Definición del Espacio ---
espacio_dimension = 100 # Un cubo de 100x100x100 unidades

# --- 2. Crear Múltiples Neuronas con Coordenadas 3D ---
num_neuronas = 50
neuronas = {}
for i in range(num_neuronas):
    x = np.random.uniform(0, espacio_dimension)
    y = np.random.uniform(0, espacio_dimension)
    z = np.random.uniform(0, espacio_dimension)
    neuronas[f'N_{i}'] = {'pos': np.array([x, y, z]), 'tipo': random.choice(['excitatoria', 'inhibitoria'])}

# --- 3. Mecanismo de Conectividad (Regla simple: por distancia) ---
G = nx.DiGraph() # Grafo dirigido para las conexiones sinápticas
distancia_max_conexion = 20 # Distancia máxima para que dos neuronas se conecten
prob_conexion_base = 0.1 # Probabilidad base para conectar

for id1, n1_data in neuronas.items():
    G.add_node(id1, pos=n1_data['pos'], tipo=n1_data['tipo'])
    for id2, n2_data in neuronas.items():
        if id1 == id2:
            continue

        dist = np.linalg.norm(n1_data['pos'] - n2_data['pos'])

        if dist < distancia_max_conexion:
            # La probabilidad de conexión disminuye con la distancia
            prob_conexion = prob_conexion_base * (1 - (dist / distancia_max_conexion))
            if random.random() < prob_conexion:
                # Añadir conexión (sinapsis)
                peso = np.random.uniform(0.1, 1.0) # Fuerza sináptica
                G.add_edge(id1, id2, weight=peso)

# --- 4. Simulación de la Actividad (Modelo muy simple) ---
# Aquí solo visualizaremos la red, la simulación de actividad sería el siguiente paso
# Puedes añadir lógica de activación y propagación de pulsos (spikes)

# --- 5. Visualización de las Neuronas en 3D ---
fig = plt.figure(figsize=(10, 10))
ax = fig.add_subplot(111, projection='3d')

# Dibujar neuronas
node_xyz = np.array([neuronas[node]['pos'] for node in G.nodes()])
ax.scatter(node_xyz[:, 0], node_xyz[:, 1], node_xyz[:, 2],
           c=['red' if G.nodes[node]['tipo'] == 'excitatoria' else 'blue' for node in G.nodes()],
           s=50, label='Neuronas')

# Dibujar conexiones (sinapsis)
for edge in G.edges():
    n1_pos = neuronas[edge[0]]['pos']
    n2_pos = neuronas[edge[1]]['pos']
    ax.plot([n1_pos[0], n2_pos[0]],
            [n1_pos[1], n2_pos[1]],
            [n1_pos[2], n2_pos[2]],
            color='gray', alpha=0.3)

ax.set_title("Simulación simple de neuronas en 3D")
ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.set_zlabel("Z")
ax.set_xlim(0, espacio_dimension)
ax.set_ylim(0, espacio_dimension)
ax.set_zlim(0, espacio_dimension)
plt.legend()
plt.show()

### Neuronas flotantes como grafo (traído del cuaderno 01)

Convierte palabras con embedding y posición en «neuronas flotantes» y las une en un grafo de vecinos cercanos (PyTorch Geometric).

In [ ]:
import torch
from torch.utils.data import Dataset
import pandas as pd
import numpy as np

class FloatingNeuron:
    def __init__(self, ID, position, vector768):
        self.id = ID
        self.x, self.y = position
        self.vector768 = vector768

class FloatingNeuronDataset(Dataset):
    def __init__(self, pkl_path):
        self.data = pd.read_pickle(pkl_path)
        self.neurons = []

        for row in self.data.itertuples():
            n = FloatingNeuron(
                ID=row.word,
                position=(row.x, row.y),
                vector768=torch.tensor(row.vector768, dtype=torch.float32)
            )
            self.neurons.append(n)

    def __len__(self):
        return len(self.neurons)

    def __getitem__(self, idx):
        neuron = self.neurons[idx]
        return {
            "id": neuron.id,
            "position": torch.tensor([neuron.x, neuron.y], dtype=torch.float32),
            "embedding": neuron.vector768
        }

In [ ]:
from torch_geometric.data import Data
from sklearn.neighbors import NearestNeighbors

def build_graph_from_neurons(dataset, k=5):
    num_nodes = len(dataset)
    positions = torch.stack([dataset[i]["position"] for i in range(num_nodes)])
    embeddings = torch.stack([dataset[i]["embedding"] for i in range(num_nodes)])

    # Conectamos por k vecinos más cercanos
    nbrs = NearestNeighbors(n_neighbors=k+1).fit(positions)
    distances, indices = nbrs.kneighbors(positions)

    edge_index = []

    for i in range(num_nodes):
        for j in indices[i][1:]:  # saltamos a sí mismo (primera columna)
            edge_index.append([i, j])

    edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()

    return Data(x=embeddings, pos=positions, edge_index=edge_index)

In [ ]:
# Usar todo
dataset = FloatingNeuronDataset("/content/DJ/palabras_con_embedding_y_posicion.pkl")
graph_data = build_graph_from_neurons(dataset, k=8)

print(graph_data)
# Data(x=[N, 768], pos=[N, 2], edge_index=[2, num_edges])

## Neurona con arreglo KNN

En el banco: `knn_prototipos`.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class KNearestNeighborsNeuron(nn.Module):
    """
    Una implementación conceptual de una "neurona con arreglo KNN".
    Esta neurona mantiene una memoria de 'm' patrones y su activación
    se basa en la similitud con los K vecinos más cercanos de esa memoria.
    """
    def __init__(self, input_dim, memory_size=5, k_neighbors=3, learning_rate=0.01):
        super().__init__()
        self.input_dim = input_dim
        self.memory_size = memory_size  # 'm' en nuestra discusión: cuántos patrones memoriza la neurona
        self.k_neighbors = k_neighbors  # 'K' en KNN: cuántos vecinos usa para la activación
        self.learning_rate = learning_rate

        # La "memoria" de la neurona: 'm' patrones, cada uno de 'input_dim' dimensiones
        # Inicializamos los patrones aleatoriamente
        self.register_buffer('patterns', torch.randn(memory_size, input_dim))

        # Un contador simple para el aprendizaje de "reemplazo" (opcional)
        self.register_buffer('pattern_usage_counts', torch.zeros(memory_size))


    def forward(self, x):
        """
        Calcula la activación de la neurona para una entrada 'x'.
        x: Un tensor de entrada de una sola muestra [input_dim]
           o un batch de muestras [batch_size, input_dim]
        """
        if x.dim() == 1: # Si es una sola muestra, expandir para que sea un batch de 1
            x = x.unsqueeze(0)

        # Calcular distancias euclidianas de la entrada a cada patrón en la memoria
        # x: [batch_size, input_dim]
        # self.patterns: [memory_size, input_dim]
        # Distances: [batch_size, memory_size]

        # Expandir las dimensiones para la resta y la norma
        expanded_x = x.unsqueeze(1)          # [batch_size, 1, input_dim]
        expanded_patterns = self.patterns.unsqueeze(0) # [1, memory_size, input_dim]

        # Calcular la distancia euclidiana al cuadrado (más eficiente que la raíz cuadrada)
        # torch.cdist también se puede usar, pero la forma manual es clara aquí.
        distances_sq = torch.sum((expanded_x - expanded_patterns)**2, dim=2) # [batch_size, memory_size]
        distances = torch.sqrt(distances_sq) # [batch_size, memory_size]

        # Encontrar los K vecinos más cercanos (menores distancias)
        # topk retorna (valores, índices)
        # Queremos los k_neighbors más pequeños, por eso invertimos las distancias con -
        # o simplemente usamos torch.topk(..., largest=False)

        # Asegurarse de que k_neighbors no sea mayor que memory_size
        actual_k = min(self.k_neighbors, self.memory_size)

        # Tomar los k vecinos más cercanos
        # Si k_neighbors es 0, simplemente devolver 0.0 o manejar de otra forma
        if actual_k == 0:
            return torch.zeros(x.size(0), device=x.device)

        k_nearest_distances, _ = torch.topk(distances, k=actual_k, largest=False, dim=1) # [batch_size, actual_k]

        # Calcular la activación: Inverso del promedio de las distancias (o suma inversa)
        # Una menor distancia debería significar una mayor activación
        # Podemos usar 1 / (1 + distancia) o alguna función de kernel como exp(-distancia^2)

        # Usaremos el promedio de las inversas de las distancias para una "activación" simple.
        # Evitar división por cero si la distancia es 0
        activations = 1.0 / (1.0 + k_nearest_distances) # [batch_size, actual_k]

        # Sumar o promediar las activaciones de los k vecinos
        neuron_output = torch.mean(activations, dim=1) # [batch_size]

        return neuron_output

    def learn_from_input(self, x_batch, threshold_distance=0.5):
        """
        Una función simplificada de "aprendizaje" para actualizar los patrones internos.
        No es backpropagation, sino un ajuste al estilo competitivo/prototipo.

        Para cada muestra en el batch:
        - Encuentra el patrón más cercano en la memoria.
        - Si la muestra es suficientemente diferente del patrón más cercano (o no lo activa lo suficiente),
          podríamos añadirla como un nuevo patrón o reemplazar el menos utilizado.
        - Si es muy similar, podríamos ajustar ligeramente el patrón más cercano hacia la muestra.

        Esta es una aproximación y puede ser más compleja en un modelo real.
        """
        # Asegurar que x_batch es 2D
        if x_batch.dim() == 1:
            x_batch = x_batch.unsqueeze(0)

        for i in range(x_batch.size(0)):
            x = x_batch[i]

            # Calcular distancias a todos los patrones
            distances = torch.sqrt(torch.sum((x - self.patterns)**2, dim=1)) # [memory_size]

            # Encontrar el patrón más cercano
            min_dist, closest_pattern_idx = torch.min(distances, dim=0)

            # Estrategia de aprendizaje simple:
            if min_dist > threshold_distance:
                # Si la entrada es "nueva" o lo suficientemente diferente del patrón más cercano,
                # reemplazar el patrón menos utilizado o el más antiguo en la memoria.
                least_used_idx = torch.argmin(self.pattern_usage_counts)
                self.patterns[least_used_idx] = x.clone().detach() # Reemplazar con la nueva muestra
                self.pattern_usage_counts[least_used_idx] = 1 # Reiniciar su conteo
            else:
                # Si la entrada es similar a un patrón existente, ajustar ese patrón ligeramente.
                # Esto es como el aprendizaje de Kohonen (SOM)
                self.patterns[closest_pattern_idx] = (1 - self.learning_rate) * self.patterns[closest_pattern_idx] + \
                                                      self.learning_rate * x.clone().detach()
                self.pattern_usage_counts[closest_pattern_idx] += 1 # Incrementar el uso


# --- Ejemplo de Uso ---
input_dimension = 10  # Dimensión de las entradas a esta neurona
memory_capacity = 5   # Número de patrones que la neurona puede "recordar"
k_to_use = 3          # Usar los 3 patrones más cercanos para la activación

# Crear una neurona con arreglo KNN
neuron = KNearestNeighborsNeuron(input_dim=input_dimension,
                                  memory_size=memory_capacity,
                                  k_neighbors=k_to_use)

print(f"Neurona creada con {memory_capacity} patrones en memoria y K={k_to_use} vecinos.")
print("Patrones iniciales de la neurona:\n", neuron.patterns)

# --- Simular el proceso de aprendizaje ---
print("\n--- Simulación de Aprendizaje ---")
# Generar algunas "experiencias" (datos de entrada)
num_learning_steps = 20
for step in range(num_learning_steps):
    # Crear una entrada aleatoria o un patrón específico
    if step % 5 == 0: # Simular algunas entradas "nuevas" o muy diferentes
        new_input = torch.randn(input_dimension) * 2 # Más varianza para ver el reemplazo
    else: # Entradas más cercanas a un patrón base para ver el ajuste
        new_input = torch.randn(input_dimension) * 0.1 + torch.tensor([1.0]*input_dimension) # clustering alrededor de [1,1,...,1]

    # La neurona "aprende" de esta entrada
    neuron.learn_from_input(new_input)
    # print(f"Paso {step+1}: Entrada procesada. Patrones actualizados.")


print("\nPatrones finales de la neurona después del aprendizaje:\n", neuron.patterns)
print("Conteo de uso de patrones:\n", neuron.pattern_usage_counts)

# --- Simular la inferencia (activación) ---
print("\n--- Simulación de Inferencia (Activación) ---")
test_input_1 = torch.tensor([1.0]*input_dimension) + torch.randn(input_dimension) * 0.05 # Muy similar a un patrón aprendido
test_input_2 = torch.randn(input_dimension) * 3       # Muy diferente (ruido)
test_input_3 = torch.tensor([-1.0]*input_dimension) + torch.randn(input_dimension) * 0.05 # Nuevo patrón

activation_1 = neuron(test_input_1)
activation_2 = neuron(test_input_2)
activation_3 = neuron(test_input_3)

print(f"Activación para Input 1 (similar a patrón aprendido): {activation_1.item():.4f}")
print(f"Activación para Input 2 (ruido/muy diferente): {activation_2.item():.4f}")
print(f"Activación para Input 3 (nuevo patrón distinto): {activation_3.item():.4f}")

# Simular un batch de entrada
batch_inputs = torch.randn(4, input_dimension)
batch_activations = neuron(batch_inputs)
print(f"\nActivaciones para un batch de entradas:\n {batch_activations}")

## Neurona con árbol de decisión

In [ ]:
import torch
import torch.nn as nn
import numpy as np
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

class IDTNeuron(nn.Module):
    """
    Una implementación conceptual de una "Neurona de Árbol de Decisión Integrado (IDT-Neuron)".
    Cada neurona encapsula un pequeño DecisionTreeClassifier.
    """
    def __init__(self, input_dim, output_classes=2, max_depth=3, random_state=42):
        super().__init__()
        self.input_dim = input_dim
        self.output_classes = output_classes # Por ejemplo, si el árbol interno clasifica en 2 clases (relevante/no relevante)
        self.max_depth = max_depth
        self.random_state = random_state

        # Cada neurona tiene su propio Decision Tree Classifier
        # NOTA: En un escenario real, este árbol se entrenaría localmente.
        # Aquí, lo inicializamos, y la función 'learn_locally' lo entrenará.
        self.decision_tree = DecisionTreeClassifier(max_depth=self.max_depth,
                                                    random_state=self.random_state)

        # Una pequeña "memoria" local para simular el dataset de entrenamiento interno del árbol
        # Esto es para la simulación del aprendizaje local.
        self.local_X_memory = []
        self.local_y_memory = []
        self.memory_capacity = 50 # Cuántas muestras puede "recordar" para entrenar su árbol

    def forward(self, x):
        """
        Calcula la activación de la neurona pasando la entrada a su árbol de decisión interno.
        x: Un tensor de entrada de una sola muestra [input_dim]
           o un batch de muestras [batch_size, input_dim]
        """
        if x.dim() == 1: # Si es una sola muestra, expandir para que sea un batch de 1
            x_np = x.detach().cpu().numpy().reshape(1, -1)
        else:
            x_np = x.detach().cpu().numpy()

        # Si el árbol aún no ha sido entrenado (no tiene clases_ activas), no puede predecir.
        # Esto ocurre antes de la primera llamada a learn_locally.
        if not hasattr(self.decision_tree, 'classes_') or len(self.local_X_memory) < 2:
            # Devuelve una activación neutral o un indicador de "no entrenado"
            return torch.zeros(x.size(0), device=x.device, dtype=torch.float32)

        # Usamos predict_proba para obtener una activación "suave" (probabilidad)
        # o predict para una activación "dura" (0 o 1)
        # Aquí elegimos la probabilidad de la clase positiva (índice 1) como activación.

        # [batch_size, output_classes]
        probabilities = self.decision_tree.predict_proba(x_np)

        # Asumiendo que la clase positiva es el índice 1
        # Si output_classes > 2, necesitarías decidir cómo combinar las probabilidades.
        activation = torch.tensor(probabilities[:, 1], dtype=torch.float32, device=x.device)

        return activation

    def learn_locally(self, X_local_samples, y_local_labels):
        """
        Simula el aprendizaje local de la neurona, entrenando su árbol de decisión interno.
        X_local_samples: Un batch de características de entrada para que esta neurona aprenda.
                         [num_samples, input_dim]
        y_local_labels: Etiquetas asociadas a estas muestras (el "objetivo" local de esta neurona).
                        [num_samples]
        """
        if not isinstance(X_local_samples, np.ndarray):
            X_local_samples = X_local_samples.detach().cpu().numpy()
        if not isinstance(y_local_labels, np.ndarray):
            y_local_labels = y_local_labels.detach().cpu().numpy()

        # Añadir las nuevas muestras a la memoria local de la neurona
        for x_samp, y_samp in zip(X_local_samples, y_local_labels):
            self.local_X_memory.append(x_samp)
            self.local_y_memory.append(y_samp)

            # Mantener la memoria dentro de la capacidad definida
            if len(self.local_X_memory) > self.memory_capacity:
                self.local_X_memory.pop(0) # Eliminar la muestra más antigua
                self.local_y_memory.pop(0)

        # Solo entrenar si hay suficientes muestras en la memoria
        if len(self.local_X_memory) >= 2: # Se necesitan al menos 2 muestras para entrenar un DT
            current_X = np.array(self.local_X_memory)
            current_y = np.array(self.local_y_memory)

            # Entrenar el árbol de decisión interno con la memoria acumulada
            self.decision_tree.fit(current_X, current_y)

            # Opcional: imprimir la precisión local del árbol
            # y_pred_local = self.decision_tree.predict(current_X)
            # local_acc = accuracy_score(current_y, y_pred_local)
            # print(f"  Neurona entrenada localmente. Precisión interna: {local_acc:.2f}")

    def print_internal_tree(self, feature_names=None):
        """Muestra la estructura del árbol de decisión interno de la neurona."""
        if hasattr(self.decision_tree, 'tree_') and self.decision_tree.tree_.node_count > 1:
            print(f"\n--- Estructura del Árbol de Decisión Interno de IDT-Neuron ---")
            r = export_text(self.decision_tree, feature_names=feature_names)
            print(r)
        else:
            print("\nEl árbol de decisión interno aún no ha sido entrenado o es trivial.")


# --- Ejemplo de Uso ---

# 1. Crear una neurona IDT
input_dimension = 5 # La neurona espera 5 características de entrada
neuron = IDTNeuron(input_dim=input_dimension, output_classes=2, max_depth=2)
print(f"Neurona IDT creada. Profundidad máxima del árbol interno: {neuron.max_depth}")

# 2. Simular el Aprendizaje Local para la Neurona
print("\n--- Simulación de Aprendizaje Local ---")

# Generar algunos datos de ejemplo para que la neurona aprenda localmente
# X_data: características que llegarían a esta neurona desde la capa anterior
# y_labels: la "señal de supervisión" local para esta neurona (ej. si esta entrada debe activar la neurona o no)
np.random.seed(0)
num_samples = 40
X_learning_data = np.random.rand(num_samples, input_dimension) * 10
# Crear etiquetas simples: si la primera característica es alta, la etiqueta es 1, de lo contrario 0.
y_learning_labels = (X_learning_data[:, 0] > 5).astype(int)

# Simular "batches" de aprendizaje local
for i in range(0, num_samples, 10): # Aprender en grupos de 10 muestras
    batch_X = X_learning_data[i : i+10]
    batch_y = y_learning_labels[i : i+10]
    print(f"Aprendiendo con muestras {i} a {i+len(batch_X)-1}")
    neuron.learn_locally(batch_X, batch_y)

# Mostrar el árbol interno después del aprendizaje
neuron.print_internal_tree(feature_names=[f'Feature_{i+1}' for i in range(input_dimension)])


# 3. Simular la Inferencia (Activación)
print("\n--- Simulación de Inferencia (Activación) ---")

# Crear nuevas entradas para probar la neurona
test_input_1 = torch.tensor([8.0, 2.0, 3.0, 1.0, 5.0], dtype=torch.float32) # Debería activar fuerte (Feature_1 > 5)
test_input_2 = torch.tensor([2.0, 7.0, 1.0, 9.0, 4.0], dtype=torch.float32) # Debería activar débil (Feature_1 <= 5)
test_input_3 = torch.tensor([6.0, 0.5, 0.1, 9.5, 2.0], dtype=torch.float32) # Debería activar fuerte

activation_1 = neuron(test_input_1)
activation_2 = neuron(test_input_2)
activation_3 = neuron(test_input_3)

print(f"Activación para Input 1 (Feature_1=8.0): {activation_1.item():.4f}")
print(f"Activación para Input 2 (Feature_1=2.0): {activation_2.item():.4f}")
print(f"Activación para Input 3 (Feature_1=6.0): {activation_3.item():.4f}")

# Simular un batch de entrada
batch_test_inputs = torch.tensor([
    [7.5, 1.1, 2.3, 4.5, 6.7],
    [3.1, 8.2, 5.4, 0.9, 1.2],
    [9.0, 0.1, 0.2, 0.3, 0.4]
], dtype=torch.float32)
batch_activations = neuron(batch_test_inputs)
print(f"\nActivaciones para un batch de entradas:\n {batch_activations}")

## Clasificadores clásicos (la referencia de la industria para datos en tabla)

En el banco, con Random Forest y Gradient Boosting además: `python -m laboratorio.banco --clasicos`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import classification_report, f1_score, precision_score, recall_score, accuracy_score
from matplotlib.colors import ListedColormap
import warnings

# Suprimir advertencias para una salida más limpia
warnings.filterwarnings("ignore")

# --- 1. Generar un dataset sintético para clasificación ---
# Creamos un dataset con 2 características para que sea fácil de visualizar
# y 2 clases. Intentaremos que no sea linealmente separable para desafiar a los modelos.
X, y = make_classification(n_samples=500, n_features=2, n_informative=2,
                           n_redundant=0, n_clusters_per_class=1,
                           random_state=42, class_sep=1.5) # class_sep controla la separación entre clases

# Dividir el dataset en conjuntos de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print("Forma de X_train:", X_train.shape)
print("Forma de y_train:", y_train.shape)
print("Forma de X_test:", X_test.shape)
print("Forma de y_test:", y_test.shape)

# Visualizar el dataset
plt.figure(figsize=(9, 7))
plt.scatter(X[:, 0], X[:, 1], c=y, cmap='viridis', s=50, alpha=0.8, edgecolors='k')
plt.title('Dataset Sintético para Clasificación')
plt.xlabel('Característica 1')
plt.ylabel('Característica 2')
plt.colorbar(label='Clase')
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

# --- 2. Definir los clasificadores a comparar ---
classifiers = {
    "Regresión Logística": LogisticRegression(random_state=42),
    "K-Nearest Neighbors (k=5)": KNeighborsClassifier(n_neighbors=5),
    "Support Vector Machine (Lineal)": SVC(kernel='linear', random_state=42, probability=True), # probability=True para cross_val_score con F1
    "Árbol de Decisión (Max Depth=5)": DecisionTreeClassifier(max_depth=5, random_state=42),
    "Naive Bayes (Gaussiano)": GaussianNB()
}

# --- 3. Bucle para entrenar, visualizar y evaluar cada clasificador ---

# Preparar la visualización de fronteras de decisión
# Crear una malla para dibujar las fronteras
x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.02),
                     np.arange(y_min, y_max, 0.02))

cmap_light = ListedColormap(['#FFAAAA', '#AAFFAA', '#AAAAFF']) # Colores para las regiones de decisión
cmap_bold = ListedColormap(['#FF0000', '#00FF00', '#0000FF'])   # Colores para los puntos de datos reales

results = {} # Para almacenar los resultados de la validación cruzada

print("\n--- Entrenando y Evaluando Clasificadores ---")

for name, classifier in classifiers.items():
    print(f"\n***** Modelo: {name} *****")

    # Entrenar el modelo
    classifier.fit(X_train, y_train)

    # Predecir en el conjunto de prueba
    y_pred = classifier.predict(X_test)

    # --- Visualizar fronteras de decisión ---
    Z = classifier.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)

    plt.figure(figsize=(9, 7))
    plt.pcolormesh(xx, yy, Z, cmap=cmap_light, shading='auto') # Fronteras de decisión

    # Graficar los puntos de entrenamiento
    plt.scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap=cmap_bold,
                edgecolor='k', s=20, label='Datos de Entrenamiento')
    # Graficar los puntos de prueba (opcional, para ver cómo caen)
    # plt.scatter(X_test[:, 0], X_test[:, 1], c=y_test, cmap=cmap_bold,
    #             edgecolor='k', s=20, marker='x', alpha=0.6, label='Datos de Prueba')

    plt.xlim(xx.min(), xx.max())
    plt.ylim(yy.min(), yy.max())
    plt.title(f'Frontera de Decisión: {name}')
    plt.xlabel('Característica 1')
    plt.ylabel('Característica 2')
    plt.legend()
    plt.show()

    # --- Imprimir métricas en el conjunto de prueba ---
    print("\nMétricas en el Conjunto de Prueba:")
    print(classification_report(y_test, y_pred))
    print(f"Accuracy: {accuracy_score(y_test, y_pred):.2f}")
    print(f"F1-Score (weighted): {f1_score(y_test, y_pred, average='weighted'):.2f}") # Weighted F1 para clases potencialmente desequilibradas

    # --- Realizar Validación Cruzada para una evaluación robusta ---
    # Usaremos 5-fold cross-validation y F1-score como métrica principal
    # para problemas de clasificación binaria.
    cv_scores = cross_val_score(classifier, X, y, cv=5, scoring='f1_weighted')
    results[name] = cv_scores.mean()
    print(f"\nF1-Score (Weighted) con 5-Fold Cross-Validation: {cv_scores.mean():.2f} (+/- {cv_scores.std() * 2:.2f})")
    print("-" * 50)

# --- Resumen Final de Rendimiento ---
print("\n--- Resumen de F1-Score (Weighted) en Cross-Validation ---")
for name, score in sorted(results.items(), key=lambda item: item[1], reverse=True):
    print(f"{name}: {score:.2f}")

## Neurona bayesiana (Naive Bayes)

In [ ]:
import torch
import torch.nn as nn
import numpy as np
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs # Para generar datos para la neurona interna
from sklearn.model_selection import train_test_split

class NBNNeuron(nn.Module):
    """
    Una implementación conceptual de una "Neurona con Clasificador Bayesiano Ingenuo Integrado".
    Cada neurona encapsula su propio Clasificador Bayesiano Ingenuo.
    """
    def __init__(self, input_dim, output_classes=2):
        super().__init__()
        self.input_dim = input_dim
        self.output_classes = output_classes # Número de "clases" internas que el NB de la neurona aprenderá
                                            # Por ejemplo, 2 para 'patrón A' vs 'patrón B'

        # Cada neurona tiene su propio Clasificador Bayesiano Gaussiano
        self.naive_bayes_classifier = GaussianNB()

        # Una pequeña "memoria" local para simular el dataset de entrenamiento interno del NB
        self.local_X_memory = []
        self.local_y_memory = []
        self.memory_capacity = 100 # Cuántas muestras puede "recordar" para entrenar su NB

    def forward(self, x):
        """
        Calcula la activación de la neurona pasando la entrada a su clasificador Naive Bayes interno.
        x: Un tensor de entrada de una sola muestra [input_dim]
           o un batch de muestras [batch_size, input_dim]
        """
        if x.dim() == 1: # Si es una sola muestra, expandir para que sea un batch de 1
            x_np = x.detach().cpu().numpy().reshape(1, -1)
        else:
            x_np = x.detach().cpu().numpy()

        # Verificar si el clasificador interno ha sido entrenado
        if not hasattr(self.naive_bayes_classifier, 'classes_') or len(self.local_X_memory) < 2:
            # Si no ha sido entrenado, devuelve una activación neutral
            return torch.zeros(x.size(0), device=x.device, dtype=torch.float32)

        # Usamos predict_proba para obtener una activación "suave" (probabilidad)
        # La activación de la neurona será la probabilidad de la clase positiva (índice 1).
        # [batch_size, output_classes]
        probabilities = self.naive_bayes_classifier.predict_proba(x_np)

        # Asumiendo que la clase positiva es el índice 1
        # Si output_classes > 2, podrías elegir la probabilidad de la clase más alta, o la suma de ciertas probabilidades.
        activation = torch.tensor(probabilities[:, 1], dtype=torch.float32, device=x.device)

        return activation

    def learn_locally(self, X_local_samples, y_local_labels):
        """
        Simula el aprendizaje local de la neurona, entrenando/actualizando su clasificador Naive Bayes interno.
        X_local_samples: Un batch de características de entrada para que esta neurona aprenda.
                         [num_samples, input_dim]
        y_local_labels: Etiquetas asociadas a estas muestras (el "objetivo" local de esta neurona).
                        [num_samples]
        """
        if not isinstance(X_local_samples, np.ndarray):
            X_local_samples = X_local_samples.detach().cpu().numpy()
        if not isinstance(y_local_labels, np.ndarray):
            y_local_labels = y_local_labels.detach().cpu().numpy()

        # Añadir las nuevas muestras a la memoria local de la neurona
        for x_samp, y_samp in zip(X_local_samples, y_local_labels):
            self.local_X_memory.append(x_samp)
            self.local_y_memory.append(y_samp)

            # Mantener la memoria dentro de la capacidad definida
            if len(self.local_X_memory) > self.memory_capacity:
                self.local_X_memory.pop(0) # Eliminar la muestra más antigua
                self.local_y_memory.pop(0)

        # Solo entrenar si hay suficientes muestras y al menos dos clases en la memoria
        if len(self.local_X_memory) >= 2 and len(np.unique(self.local_y_memory)) >= 2:
            current_X = np.array(self.local_X_memory)
            current_y = np.array(self.local_y_memory)

            # Entrenar el clasificador Naive Bayes interno con la memoria acumulada
            # Nota: GaussianNB puede ser entrenado de forma incremental con partial_fit
            # pero para esta simulación simple, reentrenamos con todo lo acumulado.
            self.naive_bayes_classifier.fit(current_X, current_y)

            # Opcional: imprimir la precisión local del NB
            # y_pred_local = self.naive_bayes_classifier.predict(current_X)
            # local_acc = accuracy_score(current_y, y_pred_local)
            # print(f"  Neurona entrenada localmente. Precisión interna: {local_acc:.2f}")


# --- 1. Simulación de la Neurona NBN ---
print("--- Simulación de la Neurona NBN ---")
input_dimension = 5 # La neurona espera 5 características de entrada
neuron = NBNNeuron(input_dim=input_dimension, output_classes=2)
print(f"Neurona NBN creada. Espera {input_dimension} características de entrada.")

# Generar datos para el aprendizaje local de la neurona
# Creamos dos "blobs" de datos, simulando dos tipos de patrones que la neurona podría aprender.
X_local_data, y_local_labels = make_blobs(n_samples=150, n_features=input_dimension,
                                          centers=2, cluster_std=1.0, random_state=42)

# Simular el aprendizaje local de la neurona en batches
print("\n--- Simulación de Aprendizaje Local para NBN-Neuron ---")
for i in range(0, len(X_local_data), 20): # Aprender en grupos de 20 muestras
    batch_X = X_local_data[i : i+20]
    batch_y = y_local_labels[i : i+20]
    print(f"Neurona aprendiendo con muestras {i} a {i+len(batch_X)-1}")
    neuron.learn_locally(batch_X, batch_y)

# 2. Simular la Inferencia (Activación) de la Neurona NBN
print("\n--- Simulación de Inferencia (Activación) de NBN-Neuron ---")
test_input_1 = torch.tensor([0.5, 0.5, 0.5, 0.5, 0.5], dtype=torch.float32) # Cercano al centro del primer blob
test_input_2 = torch.tensor([3.0, 3.0, 3.0, 3.0, 3.0], dtype=torch.float32) # Cercano al centro del segundo blob
test_input_3 = torch.tensor([10.0, 10.0, 10.0, 10.0, 10.0], dtype=torch.float32) # Muy diferente

activation_1 = neuron(test_input_1)
activation_2 = neuron(test_input_2)
activation_3 = neuron(test_input_3)

print(f"Activación para Input 1 (cercano al centro del blob 0): {activation_1.item():.4f}")
print(f"Activación para Input 2 (cercano al centro del blob 1): {activation_2.item():.4f}")
print(f"Activación para Input 3 (muy diferente): {activation_3.item():.4f}")

# Simular un batch de entrada
batch_test_inputs = torch.tensor([
    [0.1, 0.2, 0.3, 0.4, 0.5],
    [2.8, 3.1, 2.9, 3.0, 3.2],
    [5.0, 5.0, 5.0, 5.0, 5.0] # Un punto intermedio
], dtype=torch.float32)
batch_activations = neuron(batch_test_inputs)
print(f"\nActivaciones para un batch de entradas:\n {batch_activations}")


# --- Comparativa Justa: NBN-Neuron (como componente) vs. GaussianNB (como clasificador completo) ---
print("\n--- Comparativa Justa: NBN-Neuron vs. GaussianNB Completo ---")

# Vamos a usar el mismo dataset con el que "entrenamos" la neurona NBN.
# X_local_data y y_local_labels son los datos que la neurona "vio" y almacenó.

# Entrenar un GaussianNB estándar con el mismo dataset completo
nb_full_model = GaussianNB()
nb_full_model.fit(X_local_data, y_local_labels)

print("\nModelo GaussianNB completo entrenado con todo el dataset.")

# --- 2. Inferencia y Predicción del Modelo Completo ---
print("\n--- Inferencia con el Modelo GaussianNB Completo ---")

# Usar las mismas entradas de prueba que usamos para la NBN-Neuron
test_inputs_np = torch.stack([test_input_1, test_input_2, test_input_3]).numpy()

# Predecir probabilidades con el modelo completo
probabilities_full_model = nb_full_model.predict_proba(test_inputs_np)

print(f"Probabilidad de Clase 1 (modelo completo) para Input 1: {probabilities_full_model[0, 1]:.4f}")
print(f"Probabilidad de Clase 1 (modelo completo) para Input 2: {probabilities_full_model[1, 1]:.4f}")
print(f"Probabilidad de Clase 1 (modelo completo) para Input 3: {probabilities_full_model[2, 1]:.4f}")

# Predecir clases con el modelo completo
predictions_full_model = nb_full_model.predict(test_inputs_np)
print(f"Predicción de Clase (modelo completo) para Input 1: {predictions_full_model[0]}")
print(f"Predicción de Clase (modelo completo) para Input 2: {predictions_full_model[1]}")
print(f"Predicción de Clase (modelo completo) para Input 3: {predictions_full_model[2]}")

# Comparar con las activaciones de la NBN-Neuron
print("\n--- Comparación Directa de Activaciones ---")
print(f"NBN-Neuron Activation (Input 1): {activation_1.item():.4f} | GaussianNB Full Model Probability (Input 1, Class 1): {probabilities_full_model[0, 1]:.4f}")
print(f"NBN-Neuron Activation (Input 2): {activation_2.item():.4f} | GaussianNB Full Model Probability (Input 2, Class 1): {probabilities_full_model[1, 1]:.4f}")
print(f"NBN-Neuron Activation (Input 3): {activation_3.item():.4f} | GaussianNB Full Model Probability (Input 3, Class 1): {probabilities_full_model[2, 1]:.4f}")

## Redes genéticas

Las neuronas que más se ajustan se reemplazan por mutaciones de las que menos se ajustan. En el banco: `genetica` (tal cual), `genetica_arreglada` y `genetica_invertida`.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

# Fijar semillas para reproducibilidad
torch.manual_seed(0)
np.random.seed(0)

# --- Dataset Sintético ---
# Para un problema de regresión o clasificación más interesante
# Generamos datos 2D para una mejor visualización si es necesario.
# Pero dado que tu input_size es 4, usaremos un X de 4 dimensiones.
X_data = torch.randn(100, 4) # 100 muestras, 4 features
# La salida depende de la suma de los cuadrados de los primeros dos features
y_data = (X_data[:, 0]**2 + X_data[:, 1]**2 + X_data[:, 2] + 0.5 * torch.randn(100)).unsqueeze(1) # Salida de 1 dimensión

# Convertir a numpy para usar train_test_split de sklearn
from sklearn.model_selection import train_test_split
X_np = X_data.numpy()
y_np = y_data.numpy()

X_train_np, X_test_np, y_train_np, y_test_np = train_test_split(X_np, y_np, test_size=0.2, random_state=42)

X_train = torch.from_numpy(X_train_np).float()
y_train = torch.from_numpy(y_train_np).float()
X_test = torch.from_numpy(X_test_np).float()
y_test = torch.from_numpy(y_test_np).float()


# --- Clase de Neurona con seguimiento de ajustes ---
class Neurona(nn.Module):
    _next_id = 0 # Contador global para IDs únicos de neuronas
    def __init__(self, input_size):
        super().__init__()
        self.w = nn.Parameter(torch.randn(input_size) * 0.1)
        self.b = nn.Parameter(torch.zeros(1))
        self.ajustes = 0 # Contador de ajustes por optimizador
        self.generation = 0 # Generación en la que fue creada/evolucionada
        self.id = Neurona._next_id # Asignar un ID único a cada neurona
        Neurona._next_id += 1


    def forward(self, x):
        # Asegurarse de que x sea compatible con el producto punto
        # Si x es un lote, se esperaría un mm.Linear. Esto es para una sola muestra.
        return torch.dot(self.w, x) + self.b

    def mutate_from(self, parent1, parent2=None, sigma=0.01, generation_num=0):
        with torch.no_grad():
            if parent2:
                # Cruce y mutación: promedio de padres + ruido
                self.w.copy_(0.5 * parent1.w + 0.5 * parent2.w + torch.randn_like(self.w) * sigma)
                self.b.copy_(0.5 * parent1.b + 0.5 * parent2.b + torch.randn_like(self.b) * sigma)
            else:
                # Solo mutación de un padre
                self.w.copy_(parent1.w + torch.randn_like(self.w) * sigma)
                self.b.copy_(parent1.b + torch.randn_like(self.b) * sigma)
        self.ajustes = 0 # Resetear ajustes al ser mutada/clonada
        self.generation = generation_num # Registrar la generación de origen
        self.id = Neurona._next_id # Asignar un nuevo ID si se desea un seguimiento de "nueva entidad"
        Neurona._next_id += 1 # Incrementar el contador global de IDs


# --- Capa de neuronas personalizadas ---
class Capa(nn.Module):
    def __init__(self, input_size, num_neuronas):
        super().__init__()
        self.neuronas = nn.ModuleList([Neurona(input_size) for _ in range(num_neuronas)])
        self.initial_num_neurons = num_neuronas # Guardar el número inicial

    def forward(self, x):
        # x puede ser un lote (batch_size, input_size)
        # Cada neurona procesa cada muestra del lote
        # El resultado será (batch_size, num_neuronas)
        return torch.stack([n(sample) for n in self.neuronas for sample in x]).view(x.size(0), len(self.neuronas))


    def registrar_ajustes(self):
        for n in self.neuronas:
            # Comprobar si el gradiente existe y si tiene algún valor no cero
            # Esto indica que la neurona contribuyó a la pérdida y fue ajustada
            if n.w.grad is not None and torch.any(n.w.grad != 0):
                n.ajustes += 1
            if n.b.grad is not None and torch.any(n.b.grad != 0):
                # Opcional: registrar ajustes también por el sesgo
                pass

    def evolucionar(self, generation_num):
        print(f"\n--- Evolución de neuronas en Capa (Generación {generation_num}) ---")

        # Ordenar por ajustes (menor es mejor)
        # Asegurarse de tener al menos 2 neuronas para selección
        if len(self.neuronas) < 2:
            print("¡Advertencia! Menos de 2 neuronas para la evolución. Saltando.")
            return

        # Filtrar las neuronas que no han sido inicializadas correctamente
        # o que tienen problemas en su 'ajustes' contador
        valid_neuronas = [n for n in self.neuronas if hasattr(n, 'ajustes')]

        if not valid_neuronas:
            print("No hay neuronas válidas para evolucionar.")
            return

        # Asegurarse de que tenemos al menos 2 neuronas válidas para la selección
        if len(valid_neuronas) < 2:
            print("Insuficientes neuronas válidas para seleccionar padres. Saltando evolución.")
            return

        mejores = sorted(valid_neuronas, key=lambda n: n.ajustes)[:2]

        # Listas para auditoría
        nuevas_neuronas = []
        podadas_info = []

        for i, n in enumerate(self.neuronas):
            if n.ajustes > 10: # Criterio de poda: más de 10 ajustes
                # Poda: registrar la neurona actual
                podadas_info.append(f"Neurona ID: {n.id} (Ajustes: {n.ajustes}) podada de la posición {i}.")

                # Reemplazo: crear una nueva neurona mutada de los mejores
                nuevo = Neurona(n.w.size(0)) # Mantener el mismo input_size
                nuevo.mutate_from(mejores[0], mejores[1], generation_num=generation_num)
                nuevas_neuronas.append(nuevo)
            else:
                # Si no se poda, mantener la neurona existente
                nuevas_neuronas.append(n)

        # Actualizar la lista de neuronas de la capa
        self.neuronas = nn.ModuleList(nuevas_neuronas)

        # Imprimir resumen de la evolución
        print(f"Número de neuronas podadas/reemplazadas en esta evolución: {len(podadas_info)}")
        for info in podadas_info:
            print(f"  - {info}")
        print(f"Neuronas restantes en la capa oculta: {len(self.neuronas)}\n")


# --- Red completa con una capa oculta genética y una capa de salida ---
class RedGenetica(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.oculta = Capa(input_size, hidden_size)
        self.salida = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        # La capa oculta ahora maneja el lote
        x = self.oculta(x)
        x = F.relu(x)
        return self.salida(x)

    def registrar_ajustes(self):
        self.oculta.registrar_ajustes()

    def evolucionar(self, generation_num):
        self.oculta.evolucionar(generation_num)

# --- Entrenamiento Básico con Evolución ---
input_dim = 4
hidden_dim = 10
output_dim = 1 # Ajustado a la salida de tu dataset sintético

modelo = RedGenetica(input_size=input_dim, hidden_size=hidden_dim, output_size=output_dim)
opt = torch.optim.SGD(modelo.parameters(), lr=0.01)
criterion = nn.MSELoss() # Usamos MSE para el problema de regresión

# --- Conteo Inicial de Neuronas ---
print("--- Conteo de Neuronas al Inicio ---")
print(f"Capa de Entrada: {input_dim} neuronas (features)")
print(f"Capa Oculta: {len(modelo.oculta.neuronas)} neuronas (iniciales)")
print(f"Capa de Salida: {output_dim} neuronas")

# Historial para visualización
losses = []
neuron_adjustments_history = [] # Lista de diccionarios, uno por época
pruned_neurons_count_history = [] # Para el número de neuronas podadas por evolución

for epoch in range(300):
    # Asegúrate de que las neuronas tengan sus gradientes reiniciados
    for n in modelo.oculta.neuronas:
        if n.w.grad is not None:
            n.w.grad.zero_()
        if n.b.grad is not None:
            n.b.grad.zero_()

    # Cada época procesa todo el X_train
    modelo.train()
    opt.zero_grad()
    out = modelo(X_train)
    loss = criterion(out, y_train) # Usar y_train que tiene la misma dimensión de out
    loss.backward()

    # Registrar ajustes ANTES de que el optimizador los aplique y los gradientes se vacíen
    modelo.registrar_ajustes()
    opt.step()

    losses.append(loss.item())

    # Guardar el estado de ajustes de cada neurona
    current_adjustments = {n.id: n.ajustes for n in modelo.oculta.neuronas}
    neuron_adjustments_history.append(current_adjustments)

    if epoch == 100: # Activar la evolución en la época 100
        print("\n--- Activando la Evolución Genética de Neuronas ---")
        modelo.evolucionar(epoch)
        # Tras la evolución, resetear ajustes para la siguiente fase
        for n in modelo.oculta.neuronas:
            n.ajustes = 0 # Las nuevas neuronas inician en 0, las que se mantuvieron también

        # Registro de neuronas podadas (se actualiza dentro de evolucionar, esto es para visualización)
        pruned_neurons_count_history.append(len(modelo.oculta.neuronas)) # Actualizar el conteo de neuronas restantes

    if epoch % 50 == 0:
        modelo.eval()
        with torch.no_grad():
            test_loss = criterion(modelo(X_test), y_test)
        print(f"Epoch {epoch}, Train Loss: {loss.item():.4f}, Test Loss: {test_loss.item():.4f}")

# --- Conteo Final de Neuronas ---
print("\n--- Conteo de Neuronas al Final ---")
print(f"Capa de Entrada: {input_dim} neuronas (features)")
print(f"Capa Oculta: {len(modelo.oculta.neuronas)} neuronas (después de evolución)")
print(f"Capa de Salida: {output_dim} neuronas")


### Visualización de Resultados


# --- Visualización de la Pérdida ---
plt.figure(figsize=(12, 6))
plt.plot(losses, label='Pérdida de Entrenamiento (MSE)')
plt.title('Evolución de la Pérdida del Modelo')
plt.xlabel('Época')
plt.ylabel('Pérdida')
plt.axvline(x=100, color='r', linestyle='--', label='Evolución Activada')
plt.legend()
plt.grid(True)
plt.show()

# --- Visualización de Ajustes por Neurona (antes y después de la evolución) ---
plt.figure(figsize=(14, 7))
# Graficar ajustes de las neuronas en la época 99 (antes de la evolución)
if 99 < len(neuron_adjustments_history):
    adjustments_before_evo = neuron_adjustments_history[99]
    neuron_ids_before = list(adjustments_before_evo.keys())
    adj_counts_before = [adjustments_before_evo[n_id] for n_id in neuron_ids_before]

    plt.subplot(1, 2, 1)
    plt.bar([f"N{nid}" for nid in neuron_ids_before], adj_counts_before, color='skyblue')
    plt.title(f'Ajustes por Neurona Oculta (Época 99 - Antes de Evolución)')
    plt.xlabel('ID de Neurona')
    plt.ylabel('Número de Ajustes')
    plt.xticks(rotation=90)
    plt.axhline(y=10, color='red', linestyle='--', label='Umbral de Poda (10 ajustes)')
    plt.legend()


# Graficar ajustes de las neuronas en la época 101 (después de la evolución, recién reseteados)
# o en una época posterior para ver la tendencia de las nuevas/existentes
if 101 < len(neuron_adjustments_history):
    adjustments_after_evo = neuron_adjustments_history[101] # O elige una época más tardía, ej. 150
    neuron_ids_after = list(adjustments_after_evo.keys())
    adj_counts_after = [adjustments_after_evo[n_id] for n_id in neuron_ids_after]

    plt.subplot(1, 2, 2)
    plt.bar([f"N{nid}" for nid in neuron_ids_after], adj_counts_after, color='lightgreen')
    plt.title(f'Ajustes por Neurona Oculta (Época 101 - Después de Evolución)')
    plt.xlabel('ID de Neurona')
    plt.ylabel('Número de Ajustes')
    plt.xticks(rotation=90)
    plt.axhline(y=10, color='red', linestyle='--', label='Umbral de Poda (10 ajustes)')
    plt.legend()

plt.tight_layout()
plt.show()

# Visualización de la evolución del número de neuronas (si la poda redujera el número)
# En este caso, el número es fijo porque se reemplazan, no se eliminan.
# Si hubieras implementado reducción de neuronas, esta gráfica sería útil.
# plt.figure(figsize=(8, 5))
# plt.plot(pruned_neurons_count_history, label='Número de Neuronas Ocultas')
# plt.title('Número de Neuronas Ocultas a lo Largo de las Épocas')
# plt.xlabel('Época de Evolución') # Cuando se activó la evolución
# plt.ylabel('Número de Neuronas')
# plt.show()

print("\n--- Ejecución completada ---")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import copy
import random

# Configuraciones
INPUT_SIZE = 784  # MNIST 28x28
HIDDEN_SIZE = 32
OUTPUT_SIZE = 10
NUM_EPOCHS = 900
GENETIC_START_EPOCH = 100
GENETIC_THRESHOLD = 10
BATCH_SIZE = 64

# Datos
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# Capa Oculta Evolutiva
class EvolvableLayer(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.weights = nn.Parameter(torch.randn(out_features, in_features) * 0.01)
        self.biases = nn.Parameter(torch.zeros(out_features))

        self.adjust_count = torch.zeros(out_features)
        self.ids = [f'N{i}' for i in range(out_features)]

    def forward(self, x):
        self.last_input = x.detach()
        return F.linear(x, self.weights, self.biases)

    def update_adjust_count(self, weight_grads, bias_grads):
        with torch.no_grad():
            for i in range(self.out_features):
                if weight_grads[i].abs().sum() > 1e-5 or bias_grads[i].abs().sum() > 1e-5:
                    self.adjust_count[i] += 1

    def evolve(self):
        # Selección de las 2 mejores neuronas (menos ajustadas)
        sorted_indices = torch.argsort(self.adjust_count)
        best_idx1 = sorted_indices[0].item()
        best_idx2 = sorted_indices[1].item()

        to_replace = (self.adjust_count >= GENETIC_THRESHOLD).nonzero(as_tuple=False).view(-1)

        for idx in to_replace:
            idx = idx.item()
            parent_idx = random.choice([best_idx1, best_idx2])
            self.weights.data[idx] = self.weights.data[parent_idx] + torch.randn_like(self.weights.data[parent_idx]) * 0.02
            self.biases.data[idx] = self.biases.data[parent_idx] + torch.randn_like(self.biases.data[parent_idx]) * 0.02
            self.adjust_count[idx] = 0
            self.ids[idx] = f'N{idx}_mut'

# Red completa
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = EvolvableLayer(INPUT_SIZE, HIDDEN_SIZE)
        self.output = nn.Linear(HIDDEN_SIZE, OUTPUT_SIZE)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = F.relu(self.hidden(x))
        return self.output(x)

# Entrenamiento
model = Net()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

for epoch in range(NUM_EPOCHS):
    model.train()
    total_loss = 0
    for images, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()

        # Contador de ajustes por neurona
        model.hidden.update_adjust_count(model.hidden.weights.grad, model.hidden.biases.grad)

        optimizer.step()
        total_loss += loss.item()

    if epoch == GENETIC_START_EPOCH:
        print(f"\n[Genética Activada] Epoch {epoch}")

    if epoch > GENETIC_START_EPOCH:
        model.hidden.evolve()

    if epoch % 50 == 0:
        print(f"Epoch {epoch}, Loss: {total_loss:.4f}, Ajustes promedio: {model.hidden.adjust_count.mean():.2f}")

# Evaluación
model.eval()
correct = 0
with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        correct += (predicted == labels).sum().item()

print(f"\nPrecisión final en test: {100 * correct / len(test_dataset):.2f}%")